# ARIMA model (the [`statsmodels`](https://www.statsmodels.org/stable/index.html) package)

ARIMA(p,d,q) specification (ARIMA = AutoRegressive Integrated Moving Average)

$$
\begin{aligned}
	\Delta ^d y_t &= \left(\sum_{j=0}^m\alpha_j t^j\right)+\sum_{j=1}^p\phi_j\Delta^d y_{t-j}+u_t+\sum_{s=1}^q\theta_s u_{t-s} &
	u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where
* p - order of the autoregressive part
* d - order of integration
* q - order of the moving average part 
* $\alpha_0$ - constant (d=0) or drift (d=1)

Specification using the lag operator

$$
\begin{aligned}
	\Phi(L)(1-L)^dy_t&=\left(\sum_{j=0}^k\alpha_j t^j\right)+\Theta(L)u_t & u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where the polynomials
$$
\begin{aligned}
	\Phi(z)&=1-\phi_1z-\cdots-\phi_pz^p & \Theta(z)&=1+\theta_1z+\cdots+\theta_qz^q
\end{aligned}
$$

Alternative specification (estimated in `statsmodels`!)

$$
\begin{aligned}
	\Phi(L)(1-L)^d\left(y_t-\sum_{s=d}^{d+m}\gamma_s t^s \right)&=\Theta(L)u_t & u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$

where

$$
	\Phi(L)\left(\sum_{s=d}^{d+m}\gamma_st^s\right)=\sum_{j=0}^m\alpha_j t^j
$$

In [2]:
import numpy as np
import pandas as pd

from statsmodels.tsa.api import ARIMA
from statsmodels.stats.api import het_arch, acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_predict

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

To fit, we use the classes 
* [ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html#statsmodels.tsa.arima.model.ARIMA) (specified model)
* [ARIMAResult](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMAResults.html#statsmodels.tsa.arima.model.ARIMAResults) (fitted model)

When specifying the model, you need to set the `order` (model order) and `trend` (type of trend, if trend is included in the model) parameters

|d|Trend order $m+d$|`trend`|`trend` (alternative)|
|-|:-:|:-:|:-:|
|0|-|'n'|`[0]`|
|0|0|'c'|`[1]`|
|0|1|'ct'|`[1, 1]`|
|1|-|'n'|`[0]`|
|1|1|'t'|`[0, 1]`|

## ARIMA model for forecasting series levels

### Fitting an ARIMA of a given order

Load weekly data from the [`FRED`](https://fred.stlouisfed.org/) database on Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (Symbol [`WGS10YR`](https://fred.stlouisfed.org/series/WGS10YR)) from 2000-01-01 to 2023-12-31 and create a dataframe `y`

In [3]:
y = web.DataReader(name='WGS10YR', data_source='fred', start='2000-01-01', end='2023-12-31')

Fit an ARIMA(2,1,1) model without trend for `y`

Specification

$$
	(1-\phi_1L-\phi_2 L^2)(1-L) y_t=(1+\theta L) u_{t}
$$

In [ ]:
# model specification
mod = ARIMA(y, order=(2,1,1), trend='n', missing='drop')
# fitting the model on the data
res = mod.fit()
# output the fitting results
res.summary(alpha=0.05)

### Diagnostics of the fitted model

Basic plots

In [ ]:
res.plot_diagnostics(lags=15)

plt.show()

Test for serial correlation (Ljung-Box test). Choose the number of lags 7

In [ ]:
# degrees of freedom adjustment: number of estimated coefficients = number of parameters - 1 (-sigma2)
model_df = mod.k_params-1
# for the test, discard the first d residuals (d=mod.k_diff)
acorr_ljungbox(res.resid[mod.k_diff:] , lags=[7], model_df=model_df)

Test for heteroskedasticity (for ARCH effects)

In [ ]:
# degrees of freedom adjustment: number of estimated coefficients = number of parameters - 1 (-sigma2)
model_df = mod.k_params-1
# for the test, discard the first d residuals (d=mod.k_diff)
lm_stat, lm_pval, f_stat, f_pval = het_arch(res.resid[mod.k_diff:], nlags=7, ddof=model_df)

lm_stat, lm_pval

### Forecasting

Build a forecast for 10 periods

In [ ]:
forecasts = res.forecast(steps=10)
forecasts

forecast visualization

In [ ]:
plot_predict(res, start=len(y), end=len(y)+10, alpha=0.05)

plt.show()

In [ ]:
plt.plot(y.tail(30), label='y')
plt.plot(forecasts, label='y_pred')
plt.legend()

plt.show()

## Forecasting using log transformation and ARIMA

In [ ]:
y = web.DataReader(name='GDP', data_source='fred', start='1990-01-01')

In [ ]:
# model specification
mod = ARIMA(np.log(y), order=(1,1,1), trend=[0,1], missing='drop')
# fitting the model on the data
res = mod.fit()

In [ ]:
forecasts = np.exp( res.forecast(steps=10) )
forecasts

In [ ]:
plt.plot(y.tail(30), label='y')
plt.plot(forecasts, label='y_pred')
plt.legend()

plt.show()